In [ ]:
import os
import pandas as pd
import numpy as np

INPUT_PATH  = "../../../data/phase2/labeled_signals.parquet"
OUTPUT_PATH = "../../../data/phase2/features_for_model.parquet"

FEATURE_COLS = [
    "ema_ratio", "rsi_14", "macd_hist", "atr_14",
    "session_quality_enc", "direction_enc", "signal_valid_enc",
]
LABEL_COL = "label"
META_COLS = ["date", "s3_key", "fold", "split"]

In [ ]:
def encode_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add derived and encoded columns to df. Returns a new DataFrame.
    Input df must contain: ema_20, ema_50, rsi_14, macd_hist, atr_14,
    session_quality, direction, signal_valid, label.
    """
    out = df.copy()
    out["ema_ratio"]           = out["ema_20"] / out["ema_50"]
    out["session_quality_enc"] = out["session_quality"].map({"high": 2, "medium": 1, "low": 0})
    out["direction_enc"]       = out["direction"].map({"buy": 1, "sell": -1, "none": 0})
    out["signal_valid_enc"]    = out["signal_valid"].astype(int)
    return out


def assign_walk_forward_folds(
    df: pd.DataFrame,
    train_months: int = 6,
    test_months: int = 1,
) -> pd.DataFrame:
    """
    Assign walk-forward fold metadata to each row.

    For each fold N:
      - train: rows where date falls in [fold_start, fold_start + train_months)
      - test:  rows where date falls in [fold_start + train_months,
                                         fold_start + train_months + test_months)

    Rows that don't fall into any fold's test window get fold=-1, split="unused".
    """
    df = df.copy()
    df["date"] = pd.to_datetime(df["date"])
    df["fold"]  = -1
    df["split"] = "unused"

    min_date = df["date"].min().to_period("M")
    max_date = df["date"].max().to_period("M")

    fold_idx = 0
    cursor = min_date
    while True:
        train_start = cursor
        train_end   = cursor + train_months
        test_start  = train_end
        test_end    = train_end + test_months

        if test_end > max_date + 1:
            break

        train_mask = (
            (df["date"].dt.to_period("M") >= train_start) &
            (df["date"].dt.to_period("M") <  train_end)
        )
        test_mask = (
            (df["date"].dt.to_period("M") >= test_start) &
            (df["date"].dt.to_period("M") <  test_end)
        )

        df.loc[train_mask, "fold"]  = fold_idx
        df.loc[train_mask, "split"] = "train"
        df.loc[test_mask,  "fold"]  = fold_idx
        df.loc[test_mask,  "split"] = "test"

        fold_idx += 1
        cursor += test_months

    return df


def prepare_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Full feature prep pipeline:
    1. Drop rows with NaN label (direction=="none")
    2. Encode features
    3. Assign walk-forward folds
    4. Return DataFrame with META_COLS + FEATURE_COLS + LABEL_COL

    Does NOT scale features — scaling happens inside each fold during training
    to prevent leakage.
    """
    df = df[df["label"].notna()].copy()
    df = encode_features(df)
    df = assign_walk_forward_folds(df)
    keep = META_COLS + FEATURE_COLS + [LABEL_COL]
    return df[keep].reset_index(drop=True)

In [ ]:
raw = pd.read_parquet(INPUT_PATH)
prepped = prepare_features(raw)

print(f"Total rows after dropping direction=none: {len(prepped)}")
print(f"\nFold distribution:")
print(prepped.groupby(["fold", "split"]).size().to_string())
print(f"\nLabel distribution:\n{prepped['label'].value_counts()}")

os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
prepped.to_parquet(OUTPUT_PATH, index=False)
print(f"\nSaved to {OUTPUT_PATH}")

In [ ]:
df = pd.read_parquet(OUTPUT_PATH)
print(df[META_COLS + FEATURE_COLS + [LABEL_COL]].head(10).to_string())
print(f"\nFeature dtypes:\n{df[FEATURE_COLS].dtypes}")
print(f"\nAny NaN in features: {df[FEATURE_COLS].isna().any().any()}")